# CDCR Facility Heat Risk Index

Computes a facility-level heat risk index for 31 CDCR state prisons following the Ovienmhada (2024) / VCP environmental risk framework.

**Risk = Hazard × Exposure × Vulnerability**

All sub-components are min-max normalized 0–1 before averaging within each component. Components are multiplied to produce a raw risk score, then normalized 0–100 cross-period (current and mid-century share the same normalization denominator).

## Components

| Component | Sub-components | Source |
|---|---|---|
| **Hazard** | `heat_hazard_idx_norm` / `heat_hazard_fut_idx_norm` (equal-weight mean of days_over_90, hotnights, AQI) | `data/heat_air_hazard.csv` via `tract_geoid` |
| **Exposure** | `days_indoor_above_78f_2025`, `ratio_indoor_to_outdoor`, `uhi_normalized`, `1 - pct_units_refrigeration` | `data/indoor_outdoor_heat_2025.csv` + `data/cdcr_facilities.csv` |
| **Vulnerability** | Medical acuity (P1+P2+medium), age >50, mental health (EOP), disability (DPP), race/POC | `data/cdcr_facilities.csv` |

**Note on facility coverage:** 31 of 34 state prisons have indoor exposure data. CAC, CVSP, and FWF are excluded (no indoor/outdoor heat model data available).

**Note on UHI nulls:** CCI and PVSP have no Benz & Burney (2021) UHI data — their tracts were classified as undeveloped. Imputed with system mean across 31 facilities.

**Note on PBSP ratio outlier:** PBSP (Pelican Bay, Crescent City coast) has `ratio_indoor_to_outdoor` = 15.75, driven by very few outdoor 78°F days (~4) in that coastal climate. This is physically plausible but will score PBSP at 1.0 on this sub-component. Flagged in output.

In [1]:
import pandas as pd
import numpy as np

# Load data
cdcr = pd.read_csv('data/cdcr_facilities.csv')
hazard = pd.read_csv('data/heat_air_hazard.csv')
indoor = pd.read_csv('data/indoor_outdoor_heat_2025.csv')

print(f'cdcr_facilities rows: {len(cdcr)}')
print(f'heat_air_hazard rows: {len(hazard)}')
print(f'indoor_outdoor_heat rows: {len(indoor)}')

cdcr_facilities rows: 84
heat_air_hazard rows: 9109
indoor_outdoor_heat rows: 31


## 1. Build working dataset — 31 CDCR state prisons

In [2]:
# Filter to CDCR state prisons (has cdcr_code, not fire camp)
state_prisons = cdcr[
    cdcr['cdcr_code'].notna() &
    (cdcr['cdcr_firecamp'].fillna(False) != True)
].copy()
print(f'State prisons in cdcr_facilities: {len(state_prisons)}')

# Inner join with indoor_outdoor — this restricts to the 31 with exposure data
# (excludes CAC, CVSP, FWF which have no indoor heat model data)
df = state_prisons.merge(indoor, on='cdcr_code', how='inner', suffixes=('', '_indoor'))
print(f'After join with indoor_outdoor: {len(df)} facilities')
print(f'Facilities: {sorted(df["cdcr_code"].tolist())}')

State prisons in cdcr_facilities: 34
After join with indoor_outdoor: 31 facilities
Facilities: ['ASP', 'CAL', 'CCI', 'CCWF', 'CEN', 'CHCF', 'CIM', 'CIW', 'CMC', 'CMF', 'COR', 'CRC', 'CTF', 'FOL', 'HDSP', 'ISP', 'KVSP', 'LAC', 'MCSP', 'NKSP', 'PBSP', 'PVSP', 'RJD', 'SAC', 'SATF', 'SCC', 'SOL', 'SQ', 'SVSP', 'VSP', 'WSP']


In [3]:
# Join hazard via tract_geoid
# tract_geoid may be stored as float (e.g. 6077003406.0) — normalize to string
df['tract_geoid_str'] = df['tract_geoid'].astype(str).str.split('.').str[0]
hazard['GEOID_str'] = hazard['GEOID'].astype(str)

df = df.merge(
    hazard[['GEOID_str', 'heat_hazard_idx_norm', 'heat_hazard_fut_idx_norm', 'AQI_norm']],
    left_on='tract_geoid_str', right_on='GEOID_str', how='left'
)

print(f'Hazard join nulls: {df["heat_hazard_idx_norm"].isnull().sum()}')
print(f'heat_hazard_idx_norm range: {df["heat_hazard_idx_norm"].min():.1f} – {df["heat_hazard_idx_norm"].max():.1f}')
print(f'heat_hazard_fut_idx_norm range: {df["heat_hazard_fut_idx_norm"].min():.1f} – {df["heat_hazard_fut_idx_norm"].max():.1f}')

Hazard join nulls: 0
heat_hazard_idx_norm range: 11.2 – 88.4
heat_hazard_fut_idx_norm range: 8.8 – 93.5


## 2. Normalization helper

In [4]:
def minmax_norm(series):
    """Min-max normalize a series to 0–1 across the 31 facilities."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

## 3. Hazard component

Pre-computed in `data_sources/hazards/heat_hazard.ipynb` as equal-weight mean of:
- days_over_90_norm (Cal-Adapt, cross-period normalized)
- hotnights_norm (VCP 98th percentile, cross-period normalized)
- AQI_norm (mean of ozone, PM2.5, diesel percentiles from CalEnviroScreen)

Stored as 0–100; divide by 100 for 0–1 multiplication.

In [5]:
df['hazard_current'] = df['heat_hazard_idx_norm'] / 100
df['hazard_midcentury'] = df['heat_hazard_fut_idx_norm'] / 100

print('Hazard component (0–1):')
print(df[['cdcr_code', 'hazard_current', 'hazard_midcentury']]
      .sort_values('hazard_midcentury', ascending=False).to_string(index=False))

Hazard component (0–1):
cdcr_code  hazard_current  hazard_midcentury
      ISP        0.883547           0.934973
      COR        0.769381           0.873151
     SATF        0.769381           0.873151
     NKSP        0.724990           0.805909
      CEN        0.770581           0.799678
     KVSP        0.710581           0.792517
      CIM        0.769265           0.791611
      WSP        0.715149           0.757387
      LAC        0.636939           0.745284
      VSP        0.675521           0.741874
     CCWF        0.675521           0.741874
      CIW        0.684849           0.721059
      CAL        0.721976           0.705937
      CRC        0.633072           0.693709
     CHCF        0.571903           0.659032
      SCC        0.502738           0.618173
      FOL        0.526583           0.603863
      SAC        0.526583           0.603863
      ASP        0.554882           0.587992
      CCI        0.444634           0.539063
     MCSP        0.482670      

## 4. Exposure component

4 equal-weight sub-components, each min-max normalized 0–1:
1. `days_indoor_above_78f_2025` — direct indoor heat burden
2. `ratio_indoor_to_outdoor` — building thermal amplification
3. `uhi_normalized` — geographic urban heat island (Benz & Burney 2021)
4. `1 - pct_units_refrigeration` — inverted AC coverage (high AC = low exposure)

In [6]:
# Sub-component 1: indoor 78°F days
df['exp_indoor78'] = minmax_norm(df['days_indoor_above_78f_2025'])

# Sub-component 2: ratio indoor/outdoor
# PBSP outlier: ratio = 15.75 vs system max ~2.0 for all others
print('ratio_indoor_to_outdoor — top 5:')
print(df[['cdcr_code', 'ratio_indoor_to_outdoor']]
      .sort_values('ratio_indoor_to_outdoor', ascending=False).head(5).to_string(index=False))
df['exp_ratio'] = minmax_norm(df['ratio_indoor_to_outdoor'])

# Sub-component 3: UHI (already 0–1; impute 2 nulls with system mean)
uhi_nulls = df.loc[df['uhi_normalized'].isnull(), 'cdcr_code'].tolist()
print(f'\nuhi_normalized nulls: {uhi_nulls} — imputed with system mean')
uhi_mean = df['uhi_normalized'].mean()
df['uhi_filled'] = df['uhi_normalized'].fillna(uhi_mean)
df['exp_uhi'] = minmax_norm(df['uhi_filled'])

# Sub-component 4: inverted AC fraction
# pct_units_refrigeration is 0–1; invert so high AC = low exposure
df['ac_inverted'] = 1 - df['pct_units_refrigeration']
df['exp_noac'] = minmax_norm(df['ac_inverted'])

# Exposure score = equal-weight mean of 4 sub-components
exp_cols = ['exp_indoor78', 'exp_ratio', 'exp_uhi', 'exp_noac']
df['exposure_score'] = df[exp_cols].mean(axis=1)

print('\nExposure sub-components and score:')
print(df[['cdcr_code'] + exp_cols + ['exposure_score']]
      .sort_values('exposure_score', ascending=False).to_string(index=False))

ratio_indoor_to_outdoor — top 5:
cdcr_code  ratio_indoor_to_outdoor
     PBSP                   15.750
      CTF                    1.935
      RJD                    1.932
       SQ                    1.414
      CCI                    1.278

uhi_normalized nulls: ['PVSP', 'CCI'] — imputed with system mean

Exposure sub-components and score:
cdcr_code  exp_indoor78  exp_ratio  exp_uhi  exp_noac  exposure_score
      COR      0.987578   0.061905 0.577900    1.0000        0.656846
     PBSP      0.391304   1.000000 0.309600    0.9048        0.651426
      SOL      0.987578   0.069143 0.581500    0.8657        0.625980
     NKSP      0.832298   0.052508 0.627600    0.9630        0.618852
     SATF      0.826087   0.051810 0.600800    0.9506        0.607324
      WSP      0.844720   0.052000 0.483900    0.9848        0.591355
      CIW      0.931677   0.058095 0.605100    0.5600        0.538718
      CIM      1.000000   0.063111 1.000000    0.0000        0.515778
      CCI      0.714286  

## 5. Vulnerability component

5 equal-weight sub-components, each min-max normalized 0–1:
1. **Medical acuity** — sum of P1 + P2 + medium CCHCS risk tiers (% of facility population)
2. **Age** — % over 50
3. **Mental health** — % EOP designation
4. **Disability** — % DPP placement
5. **Race/POC** — % people of color (heat inequity + structural vulnerability)

In [7]:
# Medical acuity = P1 + P2 + medium risk
df['medical_acuity'] = (
    df['cchcs_high_risk_p1_pct_2025'] +
    df['cchcs_high_risk_p2_pct_2025'] +
    df['cchcs_medium_risk_pct_2025']
)

vuln_inputs = {
    'medical_acuity': 'medical_acuity',
    'age_over_50':    'cchcs_age_over_50_pct_2025',
    'mental_health':  'cchcs_mental_health_eop_pct_2025',
    'disability':     'cchcs_dpp_pct_2025',
    'race_poc':       'race_peopleofcolor_pct',
}

# Check for nulls
print('Vulnerability input nulls:')
for label, col in vuln_inputs.items():
    n = df[col].isnull().sum()
    print(f'  {label} ({col}): {n} nulls')

# Normalize each sub-component
vuln_norm_cols = []
for label, col in vuln_inputs.items():
    norm_col = f'vuln_{label}'
    df[norm_col] = minmax_norm(df[col])
    vuln_norm_cols.append(norm_col)

# Vulnerability score = equal-weight mean
df['vulnerability_score'] = df[vuln_norm_cols].mean(axis=1)

print('\nVulnerability sub-components and score:')
print(df[['cdcr_code'] + vuln_norm_cols + ['vulnerability_score']]
      .sort_values('vulnerability_score', ascending=False).to_string(index=False))

Vulnerability input nulls:
  medical_acuity (medical_acuity): 0 nulls
  age_over_50 (cchcs_age_over_50_pct_2025): 0 nulls
  mental_health (cchcs_mental_health_eop_pct_2025): 0 nulls
  disability (cchcs_dpp_pct_2025): 0 nulls
  race_poc (race_peopleofcolor_pct): 0 nulls

Vulnerability sub-components and score:
cdcr_code  vuln_medical_acuity  vuln_age_over_50  vuln_mental_health  vuln_disability  vuln_race_poc  vulnerability_score
     CHCF             1.000000          1.000000            0.504854         1.000000       0.090454             0.719062
      CMF             0.931385          0.810507            0.587379         0.777570       0.175051             0.656378
      RJD             0.872935          0.652908            0.618932         0.583178       0.363903             0.618371
      SAC             0.869123          0.257036            1.000000         0.267290       0.684329             0.615556
      LAC             0.803050          0.281426            0.497573         0.

## 6. Risk score

Raw Risk = Hazard × Exposure × Vulnerability

Normalized 0–100 **cross-period**: current and mid-century scores share the same min/max denominator, so they are directly comparable.

In [8]:
df['raw_risk_current']    = df['hazard_current']    * df['exposure_score'] * df['vulnerability_score']
df['raw_risk_midcentury'] = df['hazard_midcentury'] * df['exposure_score'] * df['vulnerability_score']

# Cross-period normalization: min/max taken across both periods together
all_raw = pd.concat([df['raw_risk_current'], df['raw_risk_midcentury']])
raw_min, raw_max = all_raw.min(), all_raw.max()
print(f'Raw risk range (both periods): {raw_min:.4f} – {raw_max:.4f}')

df['risk_score_current']    = (df['raw_risk_current']    - raw_min) / (raw_max - raw_min) * 100
df['risk_score_midcentury'] = (df['raw_risk_midcentury'] - raw_min) / (raw_max - raw_min) * 100

print('\nRisk scores — mid-century ranked:')
print(df[['cdcr_code', 'hazard_midcentury', 'exposure_score', 'vulnerability_score',
          'risk_score_current', 'risk_score_midcentury']]
      .sort_values('risk_score_midcentury', ascending=False)
      .round(2).to_string(index=False))

Raw risk range (both periods): 0.0004 – 0.2630

Risk scores — mid-century ranked:
cdcr_code  hazard_midcentury  exposure_score  vulnerability_score  risk_score_current  risk_score_midcentury
      COR               0.87            0.66                 0.46               88.10                 100.00
     SATF               0.87            0.61                 0.49               86.19                  97.83
      CIM               0.79            0.52                 0.53               80.38                  82.72
      CMF               0.49            0.49                 0.66               58.55                  60.93
      SAC               0.60            0.42                 0.62               51.15                  58.68
      VSP               0.74            0.36                 0.50               45.79                  50.30
      LAC               0.75            0.30                 0.56               40.49                  47.41
     NKSP               0.81            0.62  

## 7. Risk categories and output

Risk categories derived from Jenks natural breaks (k=4) on mid-century risk scores.
Labels: **Low → Moderate → High → Critical**.
Same break thresholds applied to current-period scores so both time periods are comparable.

Long format: two rows per facility (current + mid-century).
Saved to `data/CDCR_heat_risk_index.csv`.

In [9]:
import mapclassify

# Risk categories: Jenks natural breaks (k=4) on mid-century risk scores
jnb = mapclassify.NaturalBreaks(df['risk_score_midcentury'].values, k=4)
breaks = jnb.bins
labels = ['Low', 'Moderate', 'High', 'Critical']
print(f'Jenks breaks (mid-century, k=4): {[round(b, 1) for b in breaks]}')

def risk_label(score, bins=breaks, lbls=labels):
    for i, b in enumerate(bins):
        if score <= b:
            return lbls[i]
    return lbls[-1]

df['risk_category_current']    = df['risk_score_current'].apply(risk_label)
df['risk_category_midcentury'] = df['risk_score_midcentury'].apply(risk_label)

print('\nMid-century risk categories:')
print(df[['cdcr_code', 'risk_score_midcentury', 'risk_category_midcentury']]
      .sort_values('risk_score_midcentury', ascending=False).to_string(index=False))

# ── Output ──────────────────────────────────────────────────────────────────
shared_cols = [
    'cdcr_code', 'name', 'latitude', 'longitude', 'average_2025_population',
    'exposure_score', 'vulnerability_score',
    'AQI_norm', 'ratio_indoor_to_outdoor', 'days_indoor_above_78f_2025',
    'uhi_normalized',
    # vulnerability raw inputs for interpretability
    'medical_acuity', 'cchcs_age_over_50_pct_2025',
    'cchcs_mental_health_eop_pct_2025', 'cchcs_dpp_pct_2025', 'race_peopleofcolor_pct',
]

current = df[shared_cols + ['hazard_current', 'risk_score_current', 'risk_category_current']].copy()
current = current.rename(columns={
    'hazard_current': 'hazard_score',
    'risk_score_current': 'risk_score',
    'risk_category_current': 'risk_category',
})
current['time_period'] = 'current'

midcentury = df[shared_cols + ['hazard_midcentury', 'risk_score_midcentury', 'risk_category_midcentury']].copy()
midcentury = midcentury.rename(columns={
    'hazard_midcentury': 'hazard_score',
    'risk_score_midcentury': 'risk_score',
    'risk_category_midcentury': 'risk_category',
})
midcentury['time_period'] = 'midcentury'

output = pd.concat([current, midcentury], ignore_index=True)
output = output.sort_values(['cdcr_code', 'time_period']).reset_index(drop=True)

score_cols = ['hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score']
output[score_cols] = output[score_cols].round(2)

output.to_csv('data/CDCR_heat_risk_index.csv', index=False)
print(f'\nSaved {len(output)} rows to data/CDCR_heat_risk_index.csv')
print(f'Facilities: {output["cdcr_code"].nunique()}, Time periods: {output["time_period"].unique()}')
print(f'Columns: {list(output.columns)}')

summary = output[output['time_period'] == 'midcentury'][
    ['cdcr_code', 'hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score', 'risk_category']
].sort_values('risk_score', ascending=False).reset_index(drop=True)
summary.index += 1
print('\nMid-century risk ranking:')
print(summary.to_string())

Jenks breaks (mid-century, k=4): [np.float64(20.1), np.float64(36.4), np.float64(60.9), np.float64(100.0)]

Mid-century risk categories:
cdcr_code  risk_score_midcentury risk_category_midcentury
      COR             100.000000                 Critical
     SATF              97.831787                 Critical
      CIM              82.723180                 Critical
      CMF              60.928810                     High
      SAC              58.684072                     High
      VSP              50.298276                     High
      LAC              47.406933                     High
     NKSP              44.915170                     High
      SOL              44.677425                     High
      CIW              42.585753                     High
     KVSP              41.263580                     High
      RJD              40.888178                     High
      WSP              40.061330                     High
     CCWF              36.415919                 Mo

## 8. PBSP ratio outlier check

In [10]:
# PBSP has ratio_indoor_to_outdoor = 15.75 — all other facilities are < 2.0
# Check how much this outlier inflates PBSP's exposure score vs. a capped version

pbsp = df[df['cdcr_code'] == 'PBSP'].iloc[0]
print(f'PBSP ratio: {pbsp["ratio_indoor_to_outdoor"]}')
print(f'PBSP outdoor 78F days (implied): {pbsp["days_indoor_above_78f_2025"] / pbsp["ratio_indoor_to_outdoor"]:.1f}')
print(f'PBSP exposure_score (with outlier): {pbsp["exposure_score"]:.3f}')
print(f'PBSP exp_ratio (with outlier): {pbsp["exp_ratio"]:.3f}')

# What would PBSP exposure score be if ratio were capped at p95 of other facilities?
other_ratios = df.loc[df['cdcr_code'] != 'PBSP', 'ratio_indoor_to_outdoor']
p95 = other_ratios.quantile(0.95)
print(f'\n95th pctl ratio (excl. PBSP): {p95:.3f}')
print('(No cap applied — outlier retained. Consider sensitivity analysis if PBSP rank is influential.)')

PBSP ratio: 15.75
PBSP outdoor 78F days (implied): 4.0
PBSP exposure_score (with outlier): 0.651
PBSP exp_ratio (with outlier): 1.000

95th pctl ratio (excl. PBSP): 1.699
(No cap applied — outlier retained. Consider sensitivity analysis if PBSP rank is influential.)
